# External bridges trade-off (coverage vs ambiguity vs runtime)

This notebook quantifies a core IDTrack design claim:

- External namespaces can **increase connectivity** (reduce 1→0), but they can also **amplify ambiguity** (increase 1→n) and runtime.
- IDTrack treats external inclusion as a **reviewable contract** (YAML allowlist) rather than a hidden default.

## Rationale

This experiment is a marketing-friendly way to justify why IDTrack uses a curated external allowlist:

- **Curated externals** are a controlled bridge that can reconnect otherwise disconnected Ensembl histories.
- **Unbounded externals** often add highly connected identifiers that explode transitive ambiguity, increasing 1→n and cost.

## What to report (Methods-facing knobs)

- Snapshot release (graph boundary)
- YAML allowlist (external contract)
- Strategy (`best` vs `all`) for ambiguity exposure

We compare two graph configurations for the same organism/snapshot:

1. **Curated externals** (`narrow_external=True`): uses the configured YAML to include only selected external databases.
2. **All externals** (`narrow_external=False`): includes all external xref tables available in Ensembl (often much noisier).

Outputs:
- Figure: `idtrack-manuscript/figures/fig_external_bridges_tradeoff.pdf`
- Figure: `idtrack-manuscript/figures/fig_external_bridges_tradeoff_pareto.pdf`
- Table: `idtrack-manuscript/tables/external_bridges_tradeoff_repeats.csv`

Caching:
- Results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/external_bridges_tradeoff/`.
- If caches are missing, the notebook computes them (graph loading can be memory-intensive).

## Interpretation guide

- Panel 1: how the outcome profile shifts (coverage vs ambiguity).
- Panel 2: throughput impact of widening external scope.
- Panel 3: the measured expansion of external scope (DBs/nodes).


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    notebook_context,
    read_pickle,
    save_figure,
    write_pickle,
)

ctx = notebook_context('external_bridges_tradeoff', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)



In [ ]:
# -------------------- Configuration --------------------

ORGANISM_ALIAS = 'human'
SNAPSHOT_RELEASE = 114
TO_RELEASE = 107
FINAL_DATABASE = 'HGNC Symbol'
STRATEGY = 'all'

N_QUERIES = 500
N_REPEATS = 5
BASE_SEED = 0
SEEDS = list(range(BASE_SEED, BASE_SEED + N_REPEATS))

RESULTS_PKL = CACHE_DIR / (
    f"tradeoff_{ORGANISM_ALIAS}_snapshot{SNAPSHOT_RELEASE}_to{TO_RELEASE}_final{FINAL_DATABASE}_"
    f"strategy{STRATEGY}_n{N_QUERIES}_reps{N_REPEATS}_seed{BASE_SEED}.pickle"
)

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import numpy as np

if RESULTS_PKL.exists():
    payload = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
else:
    import idtrack
    from idtrack import DB
    from idtrack._track import Track

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = int(SNAPSHOT_RELEASE)
    if snapshot > int(latest):
        raise ValueError(f'snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}')

    # Build a DatabaseManager once (shared across Track instances)
    dm = api.get_database_manager(organism_name=organism, snapshot_release=snapshot)

    def build_track(narrow_external: bool) -> Track:
        return Track(dm, narrow=True, narrow_external=narrow_external)

    def reservoir_sample(nodes, prefix: str, k: int, rng: np.random.Generator) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    # Build both track variants once; reuse across repeats.
    tracks = {
        'curated': build_track(narrow_external=True),
        'extall': build_track(narrow_external=False),
    }

    # Use the curated graph to pick queries (ENS nodes are the same across modes)
    api.track = tracks['curated']

    def graph_stats(track: Track) -> dict:
        g = track.graph
        n_nodes = int(g.number_of_nodes())
        n_edges = int(g.number_of_edges())
        n_ext_dbs = int(len(getattr(g, 'available_external_databases', [])))
        n_ext_nodes = 0
        for _n, att in g.nodes(data=True):
            if att.get(DB.node_type_str) == DB.nts_external:
                n_ext_nodes += 1
        return {
            'n_nodes': n_nodes,
            'n_edges': n_edges,
            'n_external_dbs': n_ext_dbs,
            'n_external_nodes': int(n_ext_nodes),
        }

    def run_once(mode: str, track: Track, queries: list[str], seed: int) -> dict:
        api.track = track

        stats = graph_stats(track)

        t0 = time.perf_counter()
        matchings = api.convert_identifier_multiple(
            queries,
            to_release=int(TO_RELEASE),
            final_database=FINAL_DATABASE,
            strategy=STRATEGY,
            verbose=True,
            pbar_prefix=f"{mode}:seed{seed}",
            explain=False,
        )
        dt = time.perf_counter() - t0

        bins = api.classify_multiple_conversion(matchings)
        n = len(bins['input_identifiers'])

        one0 = len(bins['matching_1_to_0'])
        one1 = len(bins['matching_1_to_1']) + len(bins['alternative_target_1_to_1'])
        onen = len(bins['matching_1_to_n']) + len(bins['alternative_target_1_to_n'])
        changed = len(bins['changed_only_1_to_1']) + len(bins['changed_only_1_to_n'])

        return {
            'mode': str(mode),
            'seed': int(seed),
            **stats,
            'n_queries': int(n),
            'seconds': float(dt),
            'it_per_s': float(n / dt) if dt else float('nan'),
            'frac_1_to_0': float(one0 / n) if n else float('nan'),
            'frac_1_to_1': float(one1 / n) if n else float('nan'),
            'frac_1_to_n': float(onen / n) if n else float('nan'),
            'frac_changed': float(changed / n) if n else float('nan'),
        }

    rows = []
    for seed in SEEDS:
        rng = np.random.default_rng(int(seed))
        queries = reservoir_sample(tracks['curated'].graph.nodes, 'ENSG', N_QUERIES, rng)
        if not queries:
            raise RuntimeError('No ENSG nodes found; cannot run trade-off experiment.')
        for mode, track in tracks.items():
            rows.append(run_once(mode, track, queries, int(seed)))
    payload = {
        'params': {
            'organism_alias': ORGANISM_ALIAS,
            'snapshot_release': snapshot,
            'to_release': int(TO_RELEASE),
            'final_database': FINAL_DATABASE,
            'strategy': STRATEGY,
            'n_queries': int(N_QUERIES),
            'n_repeats': int(N_REPEATS),
            'seeds': list(map(int, SEEDS)),
        },
        'rows': rows,
    }

    write_pickle(payload, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

payload


In [ ]:
# -------------------- Plot trade-off figure --------------------

rows = pd.DataFrame(payload['rows']).set_index('mode')
rows


In [ ]:
# -------------------- Export figure --------------------

df = pd.DataFrame(payload['rows'])

modes = ['curated', 'extall']

# Aggregate across repeats (mean ± std)
mean = df.groupby('mode').mean(numeric_only=True)
std = df.groupby('mode').std(numeric_only=True)

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2), constrained_layout=True)
ax0, ax1, ax2, ax3 = axes.ravel()

# Outcome fractions (grouped bars with error bars)
x = np.arange(len(modes))
width = 0.25
for i, (col, color, label) in enumerate(
    [
        ('frac_1_to_0', MANUSCRIPT_COLORS['1→0'], '1→0'),
        ('frac_1_to_1', MANUSCRIPT_COLORS['1→1'], '1→1'),
        ('frac_1_to_n', MANUSCRIPT_COLORS['1→n'], '1→n'),
    ]
):
    y = mean[col].reindex(modes).values
    yerr = std[col].reindex(modes).fillna(0).values
    ax0.bar(x + (i - 1) * width, y, width, yerr=yerr, capsize=3, color=color, label=label)
ax0.set_xticks(x)
ax0.set_xticklabels(modes)
ax0.set_ylim(0, 1)
ax0.set_ylabel('Fraction of queries')
ax0.set_title('Coverage vs ambiguity trade-off')
ax0.legend(frameon=True, title='Outcome')

# Runtime (it/s)
y = mean['it_per_s'].reindex(modes).values
yerr = std['it_per_s'].reindex(modes).fillna(0).values
ax1.bar(modes, y, yerr=yerr, capsize=3, color=MANUSCRIPT_COLORS['1→1'])
ax1.set_ylabel('Iterations per second (it/s)')
ax1.set_title('Runtime throughput (mean ± std)')

# Drift fraction (changed-only)
if 'frac_changed' in mean.columns:
    y = mean['frac_changed'].reindex(modes).values
    yerr = std['frac_changed'].reindex(modes).fillna(0).values
    ax2.bar(modes, y, yerr=yerr, capsize=3, color=MANUSCRIPT_COLORS['neutral'])
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Fraction changed')
    ax2.set_title('Drift signal (mean ± std)')
else:
    ax2.axis('off')
    ax2.text(0.5, 0.5, 'No drift metric', ha='center', va='center')

# External scope (graph stats; constant across repeats)
scope = mean[['n_external_dbs', 'n_external_nodes']].reindex(modes)
scope.plot(kind='bar', ax=ax3)
ax3.set_title('External scope')
ax3.set_ylabel('Count')
ax3.legend(['# external DBs', '# external nodes'], frameon=True)

written = save_figure(fig, 'fig_external_bridges_tradeoff.pdf', ctx, formats=('pdf',))
print('Saved:', written['pdf'])


# Marketing extension: Pareto-style trade-off scatter

The bar charts summarize mean±std. A complementary view is a **Pareto scatter** across repeats:

- x-axis: failure fraction (1→0)
- y-axis: ambiguity fraction (1→n)
- point size: throughput (it/s)

This makes the “coverage vs ambiguity vs runtime” trade-off visual in a single panel.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv, label_panels  # noqa: E402

df = pd.DataFrame(payload['rows'])
EXPERIMENT_TABLES = (ctx.experiment_outputs / 'tables')
EXPERIMENT_TABLES.mkdir(parents=True, exist_ok=True)

out_csv = ctx.manuscript_tables / 'external_bridges_tradeoff_repeats.csv'
atomic_write_dataframe_csv(df, out_csv, index=False)
atomic_write_dataframe_csv(df, EXPERIMENT_TABLES / out_csv.name, index=False)
print('Wrote:', out_csv)

figP, axP = plt.subplots(1, 1, figsize=(7.2, 4.6), constrained_layout=True)
for mode, color in [('curated', MANUSCRIPT_COLORS['1→1']), ('extall', MANUSCRIPT_COLORS['1→n'])]:
    sub = df[df['mode'] == mode]
    if sub.empty:
        continue
    sizes = 20 + 80 * (sub['it_per_s'] / sub['it_per_s'].max())
    axP.scatter(
        sub['frac_1_to_0'],
        sub['frac_1_to_n'],
        s=sizes,
        alpha=0.8,
        label=mode,
        color=color,
        edgecolor='white',
        linewidth=0.5,
    )

# Mark the means
m = df.groupby('mode', as_index=False)[['frac_1_to_0', 'frac_1_to_n', 'it_per_s']].mean(numeric_only=True)
for _, r in m.iterrows():
    axP.scatter(r['frac_1_to_0'], r['frac_1_to_n'], s=140, marker='X', color='black')
    axP.text(r['frac_1_to_0'], r['frac_1_to_n'], f"  mean({r['mode']})", va='center')

axP.set_xlabel('1→0 fraction (lower is better)')
axP.set_ylabel('1→n fraction (ambiguity)')
axP.set_title('External-bridge scope: coverage vs ambiguity vs runtime')
axP.set_xlim(0, 1)
axP.set_ylim(0, 1)
axP.legend(frameon=True)

writtenP = save_figure(figP, 'fig_external_bridges_tradeoff_pareto.pdf', ctx, formats=('pdf',))
print('Saved:', writtenP['pdf'])
